# RNA QC workflow
### Author: Kaili Fan

## Introduction

This notebook provides a reproducible workflow for running the RNA QC pipeline. It takes you from cell ranger filtered h5 files through cell quality control, doublet removal, clustering, merging, and UMAP visualization.


In this guide, we use a multi-tissue 10X Multiome dataset as an example. Due to the specificity of different tissues, we process the data on a per-tissue basis—performing QC and filtering for each sample individually. Finally, we integrate all samples from the same tissue for a sanity check and further analysis.

In [1]:
import sys
import os

repo_root = os.path.abspath(os.path.join("..")) 
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

It is recommended to start the workflow from the multi-Omic_workflow/RNA_QC directory. This ensures that all relative paths and function imports work as intended. Alternatively, you may use absolute paths if preferred.

In [2]:
os.getcwd()

'/path/to/Multiomic-workflow/RNA_QC/notebooks'

In [3]:
os.chdir('../')
os.getcwd()

'/path/to/Multiomic-workflow/RNA_QC'

All log outputs are stored in log files within the logs folder.

In [4]:
os.makedirs("test_output/logs", exist_ok=True)

## Config overview

#### YAML File for Storing Input Parameters

All parameters and file paths used in this workflow are stored in config/rna_qc_config.yaml

Please review this configuration file to ensure that all parameters are correctly specified and as expected before proceeding with the analysis.

In [5]:
from rna_qc.utils import load_config

config_path = "config/rna_qc_config.yaml"
config = load_config(config_path)
import yaml
print(yaml.dump(config, default_flow_style=False))

color:
  donor_colors:
    DonorA: '#FBB30B'
    DonorB: '#8e02da'
  tissue_colors:
    Adipose - Subcutaneous: '#1d8c46'
    Adrenal Gland: '#33DD33'
    Kidney: '#a32015'
    Liver - Left Lobe: '#AABB66'
    Lung - Left Cranial Lobe: '#99FF00'
    Muscle - Gastrocnemius: '#9191ff'
    Muscle - Quadriceps: '#5e5eff'
    Thyroid: '#006600'
my_color_palette:
- '#c50909'
- '#ffa806'
- '#fbe95d'
- '#16b900'
- '#65c4ff'
- '#0e3aa9'
- '#8e02da'
- '#795548'
- '#9E9E9E'
- '#ff5b13'
- '#154f11'
- '#8bc34a'
- '#ff37a6'
- '#0062ff'
- '#00bfae'
- '#ea9c28'
- '#6e7f27'
- '#43e8e6'
- '#a4ff5a'
- '#e72c80'
params:
  suffix: Muscle_Quadriceps
  tissue: Muscle - Quadriceps
paths:
  input_dir: /path/to/Multiomic-workflow/RNA_QC/test_data/cellranger
  output_h5ad_dir: /path/to/Multiomic-workflow/RNA_QC/test_output/rna_h5ad
  sample_metadata: /path/to/Multiomic-workflow/ATAC_QC/test_data/test_metatable.txt
  scrinvex_dir: /path/to/Multiomic-workflow/RNA_QC/test_data/scrinvex
  workdir: /path/to/Multiomic

Make sure your files are formatted similarly to those in the test_data/ directory, and update the file paths in config/rna_qc_config.yaml to point to your data files. 

You can also modify the reference files in the configuration to match your dataset if needed.

- If you want to process all samples in sample_metatable, set tissue: --- in the config file; otherwise, specify a tissue to work on.
- 'suffix' is used for file naming. Please make sure the strings do not contain spaces or special characters.
- You can specify your own tissue or donor colors in the config file if desired; otherwise, the script will automatically assign colors.

#### Table for Sample Information
All sample information is stored in the metatable, which is used throughout all sections of the workflow. You may use a comprehensive metatable containing all of your data; however, this workflow will process samples per designated tissue.

Please ensure your metatable follows the required format and includes all necessary sample data. Below is an example of how this table should look:

In [6]:
import pandas as pd

sample_metadata = pd.read_csv('test_data/test_metatable.txt', sep='\t', header=None)

sample_metadata.head()

,0,1,2,3,4,5,6
0,DonorA-GEX-01,DonorA-ATAC-01,Human,DonorA,Toddler,M,Muscle - Quadriceps
1,DonorB-GEX-02,DonorB-ATAC-02,Human,DonorB,Pre-adult,M,Muscle - Quadriceps


## Step 0 - Check CellRanger QC Summary (Optional)


This is an optional step.

Here, we summarize and visualize CellRanger QC results for each donor/sample to facilitate initial data quality assessment.

In [7]:
!python scripts/check_cellranger_qc_summary.py config/rna_qc_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/check_cellranger_qc_summary.py config/rna_qc_config.yaml > test_output/logs/check_cellranger_qc_summary.log 2>&1

2026-10-02 15:18:56 [INFO] Working tissue: Muscle - Quadriceps
2026-10-02 15:18:56 [INFO] Samples: ['DonorA-GEX-01', 'DonorB-GEX-02']
2026-10-02 15:18:56 [INFO] Loading CellRanger summaries for 2 samples...
2026-10-02 15:18:56 [INFO] Plotting general QC metrics...
2026-10-02 15:18:57 [INFO] Plotting read mappability metrics...
2026-10-02 15:18:58 [INFO] QC summary plots saved in /path/to/Multiomic-workflow/RNA_QC/test_output/cellranger_qc


## Step 1 - Concatenate h5ad files by tissue
This step combines CellRanger-filtered h5ad files from all samples into a single AnnData object for the specified tissue. This concatenation facilitates across-sample comparisons in downstream quality control and analysis.


Note: The actual QC filtering is still performed separately on each sample prior to this concatenation step.

In [8]:
!python scripts/concat_h5ad.py config/rna_qc_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/concat_h5ad.py config/rna_qc_config.yaml > test_output/logs/concat_h5ad.log 2>&1

2026-10-02 15:19:05 [INFO] ========== Processing tissue: Muscle - Quadriceps ==========
2026-10-02 15:19:05 [INFO] Processing sample: DonorA-GEX-01 (donor: DonorA, batch: 01, chanel: 01)...
2026-10-02 15:19:05 [INFO] Loading Scrinvex file for exon_reads% ...
2026-10-02 15:19:10 [INFO] Loaded legacy scrinvex for sample DonorA-GEX-01.
2026-10-02 15:19:10 [INFO] donor color for DonorA is #FBB30B
2026-10-02 15:19:10 [INFO] tissue color for Muscle - Quadriceps is #5e5eff
2026-10-02 15:19:10 [INFO] Processing sample: DonorB-GEX-02 (donor: DonorB, batch: 01, chanel: 02)...
2026-10-02 15:19:11 [INFO] Loading Scrinvex file for exon_reads% ...
2026-10-02 15:19:27 [INFO] Loaded legacy scrinvex for sample DonorB-GEX-02.
2026-10-02 15:19:27 [INFO] donor color for DonorB is #8e02da
2026-10-02 15:19:27 [INFO] tissue color for Muscle - Quadriceps is #5e5eff
2026-10-02 15:19:27 [INFO] Concatenating AnnData objects for tissue Muscle_Quadriceps...
2026-10-02 15:19:27 [INFO] Saving concatenated AnnData to

## Step 2 - Pre-QC assessment
This step summarizes and visualizes quality control (QC) metrics for each sample prior to filtering. Reviewing these metrics before filtering helps assess overall data quality and informs the selection of appropriate QC thresholds for downstream analysis.

A variety of QC metrics are visualized for each sample, and a selected subset of these metrics is used to perform the actual filtering. This ensures that only high-quality cells and genes are retained for downstream analysis.

Note: Red dashed lines on the violin plots indicate the default cutoffs, while blue dashed lines indicate the MAD-based cutoffs (default = 5 MADs from the median, but this value can be customized via input parameters). If no blue line is shown, it means all cells fall within the MAD window.

In [9]:
!python scripts/pre_qc_assessment.py config/rna_qc_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/pre_qc_assessment.py config/rna_qc_config.yaml > test_output/logs/pre_qc_assessment.log 2>&1

# or you can specify nmads by:
#!python scripts/pre_qc_assessment.py config/rna_qc_config.yaml --nmads 6.0

2026-10-02 15:19:36 [INFO] ========== Processing tissue: Muscle - Quadriceps ==========
2026-10-02 15:19:36 [INFO] Loading raw anndata obejct for tissue: Muscle_Quadriceps...
2026-10-02 15:19:36 [INFO] all figure plots by sampleID
2026-10-02 15:19:36 [INFO] use colors: {'DonorA-GEX-01': '#c50909', 'DonorB-GEX-02': '#ffa806'}
2026-10-02 15:19:40 [INFO] Plotting violin for: log10_n_genes_by_counts...
/path/to/Multiomic-workflow/RNA_QC/rna_qc/rna_plots.py:141: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
2026-10-02 15:19:41 [INFO] Plotting violin for: log10_total_counts...
/path/to/Multiomic-workflow/RNA_QC/rna_qc/rna_plots.py:141: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
2026-10-02 15:19:41 [INFO] Plotting violin for: pct_counts_in_top_50_genes...
/path/to/Multiomic-workflo

## Step 3 - QC filtering
This is a key step where cells and genes are filtered based on data-driven or user-defined QC thresholds. We recommend starting with a set of default cutoffs and then performing additional rounds of filtering as needed, to retain as many high-quality cells as possible.


Since determining optimal cutoffs may require running the QC workflow multiple times, we use a runtag to label all output files and results. You can customize this runtag for each iteration (e.g., round1, round2). In this demo, we use demo_run as the runtag. This convention applies to all subsequent steps in the workflow as well.

In [10]:
!python scripts/filter_qc.py config/rna_qc_config.yaml demo_run

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/filter_qc.py config/rna_qc_config.yaml demo_run > test_output/logs/filter_qc.demo_run.log 2>&1

2026-10-02 15:22:16 [INFO] Loading sample metadata /path/to/Multiomic-workflow/ATAC_QC/test_data/test_metatable.txt
2026-10-02 15:22:16 [INFO] Loading QC cutoff table test_data/test_QC_cutoffs.xlsx
2026-10-02 15:22:16 [INFO] ============== Processing tissue: Muscle - Quadriceps ==============
2026-10-02 15:22:16 [INFO] Loading anndata object...
2026-10-02 15:22:16 [INFO] all figure plots by sampleID
2026-10-02 15:22:16 [INFO] use colors: {'DonorA-GEX-01': '#c50909', 'DonorB-GEX-02': '#ffa806'}
2026-10-02 15:22:16 [INFO]   DonorA-GEX-01: doublet filtering metric = doublet_probabilities
2026-10-02 15:22:16 [INFO]   DonorB-GEX-02: doublet filtering metric = doublet_probabilities
2026-10-02 15:22:16 [INFO] Generating upset plot...
2026-10-02 15:22:16 [INFO] DonorA-GEX-01
2026-10-02 15:22:17 [INFO] DonorB-GEX-02
2026-10-02 15:22:18 [INFO] Export doublet information...
2026-10-02 15:22:18 [INFO] Plotting doublet score & probability distribution for: DonorA-GEX-01...
2026-10-02 15:22:18 [INFO

#### QC Cutoff Input
QC cutoffs should be provided in a table formatted similarly to test_data/test_QC_cutoffs.xlsx. Please ensure your cutoff file follows this format so the script can correctly interpret and apply your chosen values.

Below is an example of how the QC cutoff table should look:

In [11]:
import pandas as pd

qc_cutoffs = pd.read_excel('test_data/test_QC_cutoffs.xlsx')

qc_cutoffs.head()

,rnaID,Species,donorID,Age,Gender,Tissue,Min_cells_for_genes,Min_genes_in_cells,Max_genes_in_cells,Max_counts_in_cells,Max_percent_mt_in_cells,Max_percent_ribo_in_cells,Exon_ratio_cutoffs,MALAT1_CPM_cutoffs,MALAT1_CPM_max_cutoffs,Whether_batch_correction,doublet_cutoffs
0,DonorA-GEX-01,Human,DonorA,Toddler,M,Muscle - Quadriceps,3,200,5000,25000,5,3,---,10,---,Yes,0.5
1,DonorB-GEX-02,Human,DonorB,Pre-adult,M,Muscle - Quadriceps,3,200,5000,25000,5,3,---,10,---,Yes,0.7


As a general pan-tissue analysis experience, we suggested setting up and starting with default cutoffs as:
- Min_cells_for_genes: 3
- Min_genes_in_cells: 200
- Max_genes_in_cells: 5000 for non-brain; 10000 for brain/testis
- Max_counts_in_cells: 25000
- Max_percent_mt_in_cells: 15
- Max_percent_ribo_in_cells: 10
- MALAT1_CPM_cutoffs: 10
- doublet_cutoffs: 0.5

Note: 
- If Whether_batch_correction = "Yes", Harmony batch correction will be performed.
- If Whether_use_double_GMM_method = "Yes", a GMM-based method will be used to convert the doublet score into a doublet probability, and filtering will be based on this probability. If Whether_use_double_GMM_method = "No", the raw doublet score (without GMM conversion) will be used directly for filtering.

## Step 4 - Batch correction (Optional)
This step is optional. 

In this step, we use Harmony to remove batch effects across different samples.

If your data do not display significant batch effects, you may skip this step.
Please indicate whether batch correction was performed in the QC cutoff table to ensure the workflow runs properly.

In [12]:
!python scripts/batch_correction.py config/rna_qc_config.yaml demo_run

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/batch_correction.py config/rna_qc_config.yaml demo_run > test_output/logs/batch_correction.demo_run.log 2>&1

2026-10-02 15:22:45 [INFO] ============== Processing tissue: Muscle - Quadriceps ==============
2026-10-02 15:22:45 [INFO] Loaded AnnData for tissue Muscle - Quadriceps (4364 cells, 32506 genes)
2026-10-02 15:22:45 [INFO] Running Harmony integration...
2026-10-02 15:22:45,461 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-10-02 15:22:45 [INFO] Computing initial centroids with sklearn.KMeans...
2026-10-02 15:22:46,252 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-02 15:22:46 [INFO] sklearn.KMeans initialization complete.
2026-10-02 15:22:46,290 - harmonypy - INFO - Iteration 1 of 30
2026-10-02 15:22:46 [INFO] Iteration 1 of 30
2026-10-02 15:22:47,047 - harmonypy - INFO - Iteration 2 of 30
2026-10-02 15:22:47 [INFO] Iteration 2 of 30
2026-10-02 15:22:47,641 - harmonypy - INFO - Iteration 3 of 30
2026-10-02 15:22:47 [INFO] Iteration 3 of 30
2026-10-02 15:22:48,307 - harmonypy - INFO - Iteration 4 of 30
2026-10-02 15:22:48 [INFO] Iterati

## Step 5 - Clustering and QC re-assessment
In this step, we follow standard processing procedures to analyze the post-filtered data and re-assess all QC metrics to ensure that low-quality cells have been removed. If debris or poor-quality cells are still observed, you can use the re-assessment figures provided here to adjust QC cutoffs and perform additional filtering iterations as needed.

Here, you can plot your figures by sample, donor, or each aliquot by inputting parameters as sampleID/donorID/aliquotID.

In [13]:
!python scripts/clustering_and_qc_reassessment.py config/rna_qc_config.yaml demo_run sampleID

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/clustering_and_qc_reassessment.py config/rna_qc_config.yaml demo_run sampleID > test_output/logs/clustering_and_qc_reassessment.demo_run.log 2>&1

# or you can specify nmads by:
#!python scripts/clustering_and_qc_reassessment.py config/rna_qc_config.yaml demo_run sampleID --nmads 6

2026-10-02 15:23:10 [INFO] ========== Processing tissue: Muscle - Quadriceps ==========
2026-10-02 15:23:10 [INFO] Loading anndata object...
2026-10-02 15:23:10 [INFO] adding aliquot ID...
2026-10-02 15:23:10 [INFO] all figure plots by sampleID
2026-10-02 15:23:10 [INFO] use colors: {'DonorA-GEX-01': '#c50909', 'DonorB-GEX-02': '#ffa806'}
... storing 'aliquotID' as categorical
2026-10-02 15:23:19 [INFO] Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-02 15:23:19 [INFO] Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-02 15:23:19 [INFO] Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropri



The Final product of this section will be "{Suffix}_GEX.filtered.processed.{runtag}.h5ad"

# Bash Script to Run the Entire RNA QC Workflow
If you have determined your QC and doublet cutoffs and want to process all samples in one go, you can use the provided bash script at notebooks/run_rna_qc_workflow.sh to run the entire workflow. Alternatively, you may modify the script to run only specific steps as needed.

In [ ]:
# example bash command:
nohup bash notebooks/run_rna_qc_workflow.sh > test_output/logs/run_rna_qc_workflow.log 2>&1 &